# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s for further inspection.

In [ ]:
# List all available record sets and their fields, referencing their `@id`s
for rs in metadata.record_sets:
    print(f"Record Set: {rs.id} ({rs.name if hasattr(rs,'name') else ''})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    {field.id} ({field.name if hasattr(field, 'name') else ''}) Type: {field.data_type if hasattr(field, 'data_type') else ''}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis, using the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Select all record set ids dynamically from metadata
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for Record Set: {record_set_id} with shape {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))
    print("-")

# Choose one record set for analysis
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id is not None:
    print(f"Selected record set for further analysis: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

df = dataframes[main_record_set_id].copy()
# List all numeric columns by dtype or by domain knowledge
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns available: {numeric_columns}")

# If numeric columns are not automatically detected, try to choose a plausible one manually (e.g., 'coverage_percent', 'PeptideCount', 'MW', etc.)
numeric_field = numeric_columns[0] if numeric_columns else None

# Define a threshold
threshold = 10
if numeric_field:
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize column
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical (non-numeric) field if available
    cat_columns = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    group_field = cat_columns[0] if cat_columns else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        # Show mean value per group if there are enough unique categories
        plot_data = (
            filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            .sort_values(numeric_field, ascending=False)
            .head(20)
        )
        sns.barplot(data=plot_data, x=group_field, y=numeric_field)
        plt.xticks(rotation=90)
        plt.title(f"Mean {numeric_field} by {group_field} (top 20)")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded record sets and fields as defined by Croissant schema `@id`s.
- Examined numeric and categorical data for initial filtering and normalization.
- Performed group-wise analyses and visualized the distribution of key numeric variables.

Further analyses may include detailed feature engineering, outlier detection, advanced visualizations, and linking results to biological or experimental hypotheses stated in the metadata.